In [1]:
from sklearn.model_selection import KFold, StratifiedKFold
import os
import re
import numpy as np
import pandas as pd
import torch

In [2]:
def read_split_file(path):

    with open(path, 'r') as inf:
        dict_from_file = eval(inf.read())
        
    return dict_from_file

def dataset_prep(lists, data_folder, time_steps, outcomes, endpoint):

    labels = list()
    dataset_= list()
    outcome_file = pd.read_excel(outcomes)
    list_out = outcome_file[endpoint].values
    #outcome_file[endpoint]=torch.nn.functional.one_hot(torch.as_tensor(np.array(list(outcome_file[endpoint]))), num_classes=2).float()
    transformed_out = torch.nn.functional.one_hot(torch.as_tensor(np.array(list_out)), num_classes=2).float()

    for patient in lists:

        dict_ = {}
        
        all_names = os.listdir(os.path.join(data_folder,patient))

        dict_['ct'] = os.path.join(data_folder,patient,patient+"_ct.nii.gz")

        fractions = [int(match.group(1)) for path in all_names for match in re.finditer(r'fr_(\d+)(?=_reg)', path)]
        fractions.sort()
        selected_fr = fractions[:(time_steps+1)]
        
        if time_steps>1: #if time_step
            for j in range(time_steps):
                index_ = 'ct_'+str(j+1)
                dict_[str(index_)] = os.path.join(data_folder,patient,patient+"_ct_fr_"+str(selected_fr[j])+"_reg.nii.gz")

        dict_['mask'] = os.path.join(data_folder,patient,patient+"_ct_GTVtot.nii.gz")
        dict_['ID'] = patient

        index = list(outcome_file["PatientID"]).index(patient)
        dict_['label'] = transformed_out[index]
        labels.append(list_out[index])
        
        dataset_.append(dict_)

    return dataset_ , labels

In [25]:
time_points = 4
endpoint = "DFS"
kfolds = 5
data_folder = '/scratch/hb-weeklyCT/dataLongitudinal/'
outcomes = '/scratch/hb-weeklyCT/experimentsLongitudinal/outcomes.xlsx'
split_folder = '/scratch/hb-weeklyCT/experimentsLongitudinal/data_split/'

In [26]:
split_file = os.path.join(split_folder,"data_split_seqNum_"+str(time_points)+".txt")
split_data = read_split_file(split_file)

In [27]:
split_file = os.path.join(split_folder,"data_split_seqNum_"+str(time_points)+".txt")
split_data = read_split_file(split_file)
train_list, labels = dataset_prep(split_data['train'],data_folder, time_points, outcomes, endpoint)
test_list, _ = dataset_prep(split_data['test'],data_folder, time_points, outcomes, endpoint)

In [28]:
kf = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=42)

train_ids = list()
val_ids = list()


for fold, (train_index, val_index) in enumerate(kf.split(X = train_list, y=np.array(labels))):
        
    train_data = [train_list[i] for i in train_index]
    #train_ds = Dataset(data=train_data, transform=self.train_tranforms)
    #train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=False) #changed to False
    train_ids.append([l['ID'] for l in train_data])
    
    val_data = [train_list[i] for i in val_index]
    #validate_ds = Dataset(data=val_data, transform=self.train_tranforms)
    #validate_loader = DataLoader(validate_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=False) #changed to False
    val_ids.append([l['ID'] for l in val_data])
    
    print(f"Training starting for fold {fold+1}")
    print("ratio events in training: " + str(np.sum([labels[i] for i in train_index])/len(train_index)))
    print("ratio events in validation: " + str(np.sum([labels[i] for i in val_index])/len(val_index)))

Training starting for fold 1
ratio events in training: 0.32653061224489793
ratio events in validation: 0.32
Training starting for fold 2
ratio events in training: 0.32653061224489793
ratio events in validation: 0.32
Training starting for fold 3
ratio events in training: 0.32653061224489793
ratio events in validation: 0.32
Training starting for fold 4
ratio events in training: 0.32323232323232326
ratio events in validation: 0.3333333333333333
Training starting for fold 5
ratio events in training: 0.32323232323232326
ratio events in validation: 0.3333333333333333


In [29]:
for k in val_ids:
    print(len(k))

25
25
25
24
24


In [30]:
folds_save = {}
saver = []
for k in range(len(val_ids)):
    folds_save = {}
    folds_save['train']=train_ids[k]
    folds_save['validate']=val_ids[k]
    saver.append(folds_save)

In [31]:
import json
split_folder = '/scratch/hb-weeklyCT/experimentsLongitudinal/data_split/'
split_file = os.path.join(split_folder,"train_split_seqNum_"+str(time_points)+".txt")

with open(split_file, 'w') as outfile:
    json.dump(saver, outfile)

In [76]:
#ime_points = 2
endpoint = "DFS"
#folds = 3
data_folder = '/scratch/hb-weeklyCT/dataLongitudinal/'
outcomes = '/scratch/hb-weeklyCT/experimentsLongitudinal/outcomes.xlsx'
split_folder = '/scratch/hb-weeklyCT/experimentsLongitudinal/data_split/'

split_file = os.path.join(split_folder,"data_split_seqNum_"+str(time_points)+".txt")
split_data = read_split_file(split_file)
train_list, labels = dataset_prep(split_data['train'],data_folder, time_points, outcomes, endpoint)


train_id_wei = list()
val_id_wei = list()

for fold in [1,2,3]:
            
    if endpoint == 'OS' and time_points == 4:
        train_pd = pd.read_csv('/scratch/hb-weeklyCT/wei_experiments/experimentsLongitudinal/four_weeks_OS.csv')
    elif endpoint == 'DFS' and time_points == 4:
        train_pd = pd.read_csv('/scratch/hb-weeklyCT/wei_experiments/experimentsLongitudinal/four_weeks_DFS.csv')
    elif endpoint == 'OS' and time_points == 3:
        train_pd = pd.read_csv('/scratch/hb-weeklyCT/wei_experiments/experimentsLongitudinal/three_weeks_OS.csv')
    elif endpoint == 'DFS' and time_points == 3:
        train_pd = pd.read_csv('/scratch/hb-weeklyCT/wei_experiments/experimentsLongitudinal/three_weeks_DFS.csv')
    elif endpoint == 'OS' and time_points == 2:
        train_pd = pd.read_csv('/scratch/hb-weeklyCT/wei_experiments/experimentsLongitudinal/two_weeks_OS.csv')
    elif endpoint == 'DFS' and time_points == 2:
        train_pd = pd.read_csv('/scratch/hb-weeklyCT/wei_experiments/experimentsLongitudinal/two_weeks_DFS.csv') 
    train_id = train_pd[train_pd[train_pd.columns[-1]]!=fold].PatientID.values

    train_data = [sample for sample in train_list if sample['ID'] in train_id]
    
    train_id_wei.append(train_id)
    
    val_id = train_pd[train_pd[train_pd.columns[-1]]==fold].PatientID.values
    val_data = [sample for sample in train_list if sample['ID'] in val_id]
    
    val_id_wei.append(val_id)

    print(f"Training starting for fold {fold}")
    print("ratio events in training: " + str(np.sum([i['label'][1] for i in train_data])/len(train_id)))
    print("ratio events in validation: " + str(np.sum([i['label'][1] for i in val_data])/len(val_id)))
            

Training starting for fold 1
ratio events in training: 0.19117647058823528
ratio events in validation: 0.2318840579710145
Training starting for fold 2
ratio events in training: 0.21897810218978103
ratio events in validation: 0.17647058823529413
Training starting for fold 3
ratio events in training: 0.20437956204379562
ratio events in validation: 0.20588235294117646


In [77]:
for i in range(0,3):
    print(f"Fold {i+1}")
    print("Number of training patients")
    print(len(train_id_wei[i]))
    print("Number of validation patients")
    print(len(val_id_wei[i]))
    print("Total:")
    print(len(train_id_wei[i]) + len(val_id_wei[i]))
    print("  ")

Fold 1
Number of training patients
136
Number of validation patients
69
Total:
205
  
Fold 2
Number of training patients
137
Number of validation patients
68
Total:
205
  
Fold 3
Number of training patients
137
Number of validation patients
68
Total:
205
  


In [78]:
for i in range(0,3):
    print(f"Fold {i+1}")
    print("Number of training patients")
    print(len(train_ids[i]))
    print("Number of validation patients")
    print(len(val_ids[i]))
    print("Total:")
    print(len(train_ids[i]) + len(val_ids[i]))
    print("  ")

Fold 1
Number of training patients
85
Number of validation patients
43
Total:
128
  
Fold 2
Number of training patients
85
Number of validation patients
43
Total:
128
  
Fold 3
Number of training patients
86
Number of validation patients
42
Total:
128
  
